In [0]:
#Ingesting data from the csv of Region intto a dataframe
df_region_raw = spark.read.csv("/Volumes/final_project/bronze/data/Region.csv", header=True, inferSchema=True, sep="\t")
display(df_region_raw)

In [0]:
#Changing the columns of df_region_raw to snake_case naming
df_region_clean = df_region_raw.withColumnRenamed("SalesTerritoryKey", "sales_territory_key").withColumnRenamed("Region", "region").withColumnRenamed("Country", "country").withColumnRenamed("Group", "group")
display(df_region_clean)


In [0]:
#Changing the regions from USA to United States
from pyspark.sql.functions import when, col

df_region_clean = df_region_clean.withColumn("country", 
    when(col("country") == "USA", "United States")
    .when(col("country") == "UK", "United Kingdom")
    .otherwise(col("country")) # Keep everything else the same
)
display(df_region_clean)

In [0]:
#Ensure the silver shecma exists
spark.sql("CREATE SCHEMA IF NOT EXISTS final_project.silver")
#Save DataFrame in the silver schema as a delta table
df_region_clean.write.format("delta").mode("overwrite").saveAsTable("final_project.silver.region")


In [0]:
#Display the silver table
display(spark.table("final_project.silver.region"))